# Malaria Gametocyte Phenotype Pipeline — FINAL Clean Version

Single source of truth. No dead code, no duplicate installs, no leftover
`Masked`-dataset logic. Only the validated isolated-cells pipeline plus
the EfficientNet-B1 comparison and permutation significance test.

**Run top to bottom, once, in order.** If you restart the runtime,
re-run from Part 1 -- everything needed is self-contained here.

- Part 1: Setup + data acquisition (isolated cells only)
- Part 2: Feature extraction (ResNet50, robust to corrupt files)
- Part 3: EfficientNet-B1 comparison + permutation significance test
- Part 4: Hyperparameter sweep
- Part 5: Class-balance sensitivity
- Part 6: Semi-supervised embedding + held-out validation
- Part 7: Hierarchical / soft clustering
- Part 8: Supervisor-requested diagnostics
- Part 9: All figures
- Part 10: Export everything

## Part 1 — Setup & data acquisition

In [ ]:
!pip install -q umap-learn hdbscan beautifulsoup4 lxml scikit-image tifffile kneed imagecodecs
import os, time, shutil
from pathlib import Path
from ftplib import FTP
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from PIL import Image, ImageOps
from skimage import io as skio

DATA_ROOT = Path("/content/data")
FIG_DIR = Path("/content/figures")
for d in [DATA_ROOT, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)
CONDS = ["DMSO", "TC04", "TC11", "TC39"]
print("Setup complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 37.6 MB/s eta 0:00:00
Setup complete.


In [ ]:
ACCESSION = "S-BIAD633"
FTP_HOST = "ftp.ebi.ac.uk"
ACC_SUFFIX = ACCESSION[-3:]

def list_ftp_files(path):
    ftp = FTP(FTP_HOST, timeout=60)
    ftp.login()
    ftp.cwd(path)
    names = ftp.nlst()
    ftp.quit()
    return names

FTP_BASE = None
for mode in ["fire", "nfs"]:
    candidate = f"biostudies/{mode}/S-BIAD/{ACC_SUFFIX}/{ACCESSION}/Files"
    try:
        list_ftp_files(candidate)
        FTP_BASE = candidate
        print(f"Connected using mode='{mode}'")
        break
    except Exception as e:
        print(f"mode='{mode}' failed: {e}")
assert FTP_BASE is not None, "Could not connect -- check network/accession."

def robust_download(remote_dir, fname, local_path, retries=4):
    for attempt in range(1, retries + 1):
        conn = None
        try:
            conn = FTP(FTP_HOST, timeout=60)
            conn.login()
            conn.cwd(remote_dir)
            conn.set_pasv(True)
            with open(local_path, "wb") as f:
                conn.retrbinary(f"RETR {fname}", f.write)
            conn.quit()
            return True
        except Exception as e:
            print(f"    attempt {attempt}/{retries} failed for {fname}: {e}")
            try:
                if conn: conn.close()
            except Exception:
                pass
            if os.path.exists(local_path):
                os.remove(local_path)
            time.sleep(2 * attempt)
    return False

def normalise_channel(channel, q_low=0.02, q_high=0.998):
    lo, hi = np.quantile(channel, [q_low, q_high])
    clipped = np.clip(channel, lo, hi)
    return ((clipped - lo) / (hi - lo + 1e-8) * 255).astype(np.uint8)

print("Core helpers defined: robust_download, normalise_channel")

Connected using mode='fire'
Core helpers defined: robust_download, normalise_channel


In [ ]:
# Download the Individual_cells manifest (small, authoritative, complete)
manifest_ic_local = DATA_ROOT / "file_list_primary_screen_individual_cells.tsv"
ftp = FTP(FTP_HOST, timeout=60)
ftp.login()
ftp.cwd(FTP_BASE)
with open(manifest_ic_local, "wb") as f:
    ftp.retrbinary("RETR file_list_primary_screen_individual_cells.tsv", f.write)
ftp.quit()

ic_manifest = pd.read_csv(manifest_ic_local, sep="\t")
print(f"ic_manifest loaded: {len(ic_manifest)} rows")

isolated_only = ic_manifest[ic_manifest["Filetype"] == "ISOLATED_CELL"]
for cond in CONDS:
    matches = isolated_only[isolated_only["Files"].astype(str).str.contains(cond)
                             & isolated_only["Files"].astype(str).str.contains("T20")]
    print(f"  {cond}: {len(matches)} isolated cells at T20 in manifest")

ic_manifest loaded: 4488 rows
  DMSO: 138 isolated cells at T20 in manifest
  TC04: 27 isolated cells at T20 in manifest
  TC11: 62 isolated cells at T20 in manifest
  TC39: 39 isolated cells at T20 in manifest


In [ ]:
# Download the isolated cell images themselves (skip if already on disk)
IC_SUBDIR = "Primary_Screen/Individual_cells"
full_ic_remote_dir = f"{FTP_BASE}/{IC_SUBDIR}"
CELLS_DIR = DATA_ROOT / "isolated_cells"
CELLS_DIR.mkdir(exist_ok=True)

cell_index = []
download_failures = []

for cond in CONDS:
    matches = isolated_only[isolated_only["Files"].astype(str).str.contains(cond)
                             & isolated_only["Files"].astype(str).str.contains("T20")]
    for _, row in tqdm(matches.iterrows(), total=len(matches), desc=f"Downloading {cond}"):
        fname = Path(row["Files"]).name
        local_path = CELLS_DIR / fname
        if not local_path.exists() or local_path.stat().st_size == 0:
            ok = robust_download(full_ic_remote_dir, fname, str(local_path))
            if not ok:
                download_failures.append(fname)
                continue
        cell_index.append({"crop_path": str(local_path), "treatment": cond})

print(f"\n\u2713 {len(cell_index)} isolated cells ready ({len(download_failures)} failed)")
print(pd.Series([c['treatment'] for c in cell_index]).value_counts())


✓ 266 isolated cells ready (0 failed)
DMSO    138
TC11     62
TC39     39
TC04     27
Name: count, dtype: int64


## Part 2 — Feature extraction (ResNet50, robust to corrupt files)

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input

resnet = ResNet50(weights="imagenet", include_top=False, pooling="avg")

def load_and_preprocess(path, size=224):
    raw = skio.imread(str(path))
    if raw.ndim == 2:
        raw = raw[None, ...]
    ch0 = normalise_channel(raw[0])
    ch1 = normalise_channel(raw[1]) if raw.shape[0] > 1 else np.zeros_like(ch0)
    rgb = np.stack([np.zeros_like(ch0), ch0, ch1], axis=-1)
    img = Image.fromarray(rgb).resize((size, size))
    return preprocess_input(np.array(img).astype(np.float32))

# Pre-check files, drop any empty/corrupt ones
good_records = [c for c in cell_index if Path(c["crop_path"]).exists() and Path(c["crop_path"]).stat().st_size > 0]
bad_records = [c for c in cell_index if c not in good_records]
if bad_records:
    print(f"{len(bad_records)} corrupt/empty files found -- attempting one re-download each")
    for b in bad_records:
        fname = Path(b["crop_path"]).name
        if robust_download(full_ic_remote_dir, fname, b["crop_path"]) and Path(b["crop_path"]).stat().st_size > 0:
            good_records.append(b)
        else:
            print(f"  could not recover {fname} -- excluding")

cell_index = good_records
crop_paths = [c["crop_path"] for c in cell_index]
treatment_labels = np.array([c["treatment"] for c in cell_index])
features = np.zeros((len(crop_paths), 2048), dtype=np.float32)

BATCH_SIZE = 32
extraction_failures = []
for start in tqdm(range(0, len(crop_paths), BATCH_SIZE), desc="ResNet50 features"):
    batch_paths = crop_paths[start:start + BATCH_SIZE]
    arrays, valid_idx = [], []
    for j, p in enumerate(batch_paths):
        try:
            arrays.append(load_and_preprocess(p))
            valid_idx.append(start + j)
        except Exception as e:
            print(f"  ! skipping {p}: {e}")
            extraction_failures.append(p)
    if arrays:
        preds = resnet.predict(np.stack(arrays), verbose=0)
        for k, orig_idx in enumerate(valid_idx):
            features[orig_idx] = preds[k]

success_mask = ~np.all(features == 0, axis=1)
features = features[success_mask]
treatment_labels = treatment_labels[success_mask]
cell_index = [c for c, keep in zip(cell_index, success_mask) if keep]

print(f"\n\u2713 Final feature matrix: {features.shape}")
print(pd.Series(treatment_labels).value_counts())

np.save(DATA_ROOT / "features.npy", features)
pd.DataFrame(cell_index).to_csv(DATA_ROOT / "cell_index.csv", index=False)

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
treatment_ids = le.fit_transform(treatment_labels)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
2 corrupt/empty files found -- attempting one re-download each
  could not recover Sequence_ 200917 DMSO T20 008-MaxIP.nd2_crop1 - 150 x 157 x 1 x 1 - 2 ch (double).tif -- excluding
  could not recover Sequence_ 200930 DMSO T20 013-MaxIP.nd2_crop1 - 129 x 131 x 1 x 1 - 2 ch (double).tif -- excluding


ResNet50 features:   0%|          | 0/9 [00:00<?, ?it/s]


✓ Final feature matrix: (264, 2048)
DMSO    136
TC11     62
TC39     39
TC04     27
Name: count, dtype: int64


## Part 3 — EfficientNet-B1 comparison + permutation significance test

In [ ]:
from tensorflow.keras.applications import EfficientNetB1
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import umap

effnet = EfficientNetB1(weights="imagenet", include_top=False, pooling="avg")

def load_and_preprocess_effnet(path, size=240):
    raw = skio.imread(str(path))
    if raw.ndim == 2:
        raw = raw[None, ...]
    ch0 = normalise_channel(raw[0])
    ch1 = normalise_channel(raw[1]) if raw.shape[0] > 1 else np.zeros_like(ch0)
    rgb = np.stack([np.zeros_like(ch0), ch0, ch1], axis=-1)
    img = Image.fromarray(rgb).resize((size, size))
    return effnet_preprocess(np.array(img).astype(np.float32))

crop_paths = [c["crop_path"] for c in cell_index]
effnet_features = np.zeros((len(crop_paths), 1280), dtype=np.float32)
for start in tqdm(range(0, len(crop_paths), 16), desc="EfficientNet-B1 features"):
    batch_paths = crop_paths[start:start + 16]
    batch = np.stack([load_and_preprocess_effnet(p) for p in batch_paths])
    effnet_features[start:start + len(batch_paths)] = effnet.predict(batch, verbose=0)

print(f"EfficientNet-B1 feature matrix: {effnet_features.shape}")

effnet_records = []
for nn in [5, 15, 30, 50]:
    for md_ in [0.0, 0.1, 0.3, 0.5]:
        emb = umap.UMAP(n_neighbors=nn, min_dist=md_, random_state=42).fit_transform(effnet_features)
        for k in [2, 3, 4, 6, 8]:
            cl = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(emb)
            effnet_records.append({
                "nn": nn, "md": md_, "k": k,
                "ari": adjusted_rand_score(treatment_ids, cl),
                "nmi": normalized_mutual_info_score(treatment_ids, cl),
            })

effnet_df = pd.DataFrame(effnet_records)
best_effnet = effnet_df.sort_values("ari", ascending=False).iloc[0]
print("\nTop 5 EfficientNet-B1 configurations:")
print(effnet_df.sort_values("ari", ascending=False).head(5))
print(f"\nBest EfficientNet-B1 ARI: {best_effnet['ari']:.3f} (nn={best_effnet['nn']}, md={best_effnet['md']}, k={best_effnet['k']})")
print(f"Best ResNet50 ARI (Part 4, run next): compare after that cell runs")

effnet_df.to_csv(DATA_ROOT / "effnet_sweep_results.csv", index=False)
np.save(DATA_ROOT / "effnet_features.npy", effnet_features)

27018416/27018416 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


EfficientNet-B1 features:   0%|          | 0/17 [00:00<?, ?it/s]

EfficientNet-B1 feature matrix: (264, 1280)


/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr


Top 5 EfficientNet-B1 configurations:
    nn   md  k       ari       nmi
24  15  0.0  8  0.156772  0.308691
69  50  0.1  8  0.150646  0.301421
49  30  0.1  8  0.150563  0.296618
64  50  0.0  8  0.149299  0.292922
29  15  0.1  8  0.143978  0.292905

Best EfficientNet-B1 ARI: 0.157 (nn=15.0, md=0.0, k=8.0)
Best ResNet50 ARI (Part 4, run next): compare after that cell runs


In [ ]:
# Permutation significance test on the best KNOWN ResNet50 configuration
# (UMAP nn=15, md=0.5, k-means k=3 -- established in prior runs; re-fit here for a clean, self-contained cell)
best_embedding = umap.UMAP(n_neighbors=15, min_dist=0.5, random_state=42).fit_transform(features)
best_clusters = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(best_embedding)
observed_ari = adjusted_rand_score(treatment_ids, best_clusters)
print(f"Observed ARI for best ResNet50 configuration: {observed_ari:.4f}")

rng = np.random.default_rng(123)
n_permutations = 1000
null_aris = np.array([
    adjusted_rand_score(rng.permutation(treatment_ids), best_clusters)
    for _ in tqdm(range(n_permutations), desc="Permutation test")
])

p_value = np.mean(null_aris >= observed_ari)
print(f"\n=== Permutation significance test ===")
print(f"Observed ARI: {observed_ari:.4f}")
print(f"Null mean: {null_aris.mean():.4f}, std: {null_aris.std():.4f}")
print(f"Null 95th percentile: {np.percentile(null_aris, 95):.4f}")
print(f"p-value: {p_value:.4f}  ({'SIGNIFICANT' if p_value < 0.05 else 'not significant'} at alpha=0.05)")

pd.DataFrame({"null_ari": null_aris}).to_csv(DATA_ROOT / "permutation_null_distribution.csv", index=False)

/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Observed ARI for best ResNet50 configuration: 0.2973


Permutation test:   0%|          | 0/1000 [00:00<?, ?it/s]


=== Permutation significance test ===
Observed ARI: 0.2973
Null mean: -0.0002, std: 0.0081
Null 95th percentile: 0.0156
p-value: 0.0000  (SIGNIFICANT at alpha=0.05)


## Part 4 — Hyperparameter sweep (165+ configurations)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, davies_bouldin_score
import hdbscan, itertools

def purity(y_true, y_pred):
    table = pd.crosstab(y_pred, y_true)
    return table.max(axis=1).sum() / table.values.sum()

def score_clustering(embedding, cluster_labels):
    real = cluster_labels != -1
    n_found = len(set(cluster_labels[real]))
    row = {"n_clusters_found": n_found, "pct_noise": 100 * (~real).mean()}
    if n_found >= 2:
        row["silhouette"] = silhouette_score(embedding[real], cluster_labels[real])
        row["davies_bouldin"] = davies_bouldin_score(embedding[real], cluster_labels[real])
    else:
        row["silhouette"], row["davies_bouldin"] = np.nan, np.nan
    row["ari"] = adjusted_rand_score(treatment_ids, cluster_labels)
    row["nmi"] = normalized_mutual_info_score(treatment_ids, cluster_labels)
    row["purity"] = purity(treatment_ids, cluster_labels)
    return row

records = []
for n_comp in [10, 20, 30, 50, 100]:
    emb = PCA(n_components=n_comp, random_state=42).fit_transform(features)
    for k in [2, 3, 4, 6, 8]:
        cl = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(emb)
        records.append({"reduction": f"PCA({n_comp})", "clustering": f"KMeans(k={k})", **score_clustering(emb, cl)})
    for mcs in [10, 25, 50]:
        cl = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=10).fit_predict(emb)
        records.append({"reduction": f"PCA({n_comp})", "clustering": f"HDBSCAN(mcs={mcs})", **score_clustering(emb, cl)})

for nn, md_ in itertools.product([5, 15, 30, 50], [0.0, 0.1, 0.3, 0.5]):
    emb = umap.UMAP(n_neighbors=nn, min_dist=md_, random_state=42).fit_transform(features)
    for k in [2, 3, 4, 6, 8]:
        cl = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(emb)
        records.append({"reduction": f"UMAP(nn={nn},md={md_})", "clustering": f"KMeans(k={k})", **score_clustering(emb, cl)})
    for mcs in [10, 25, 50]:
        cl = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=10).fit_predict(emb)
        records.append({"reduction": f"UMAP(nn={nn},md={md_})", "clustering": f"HDBSCAN(mcs={mcs})", **score_clustering(emb, cl)})

for perp in [5, 15, 30, 50]:
    emb = TSNE(n_components=2, perplexity=perp, random_state=42, init="pca").fit_transform(features)
    for k in [2, 3, 4, 6, 8]:
        cl = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(emb)
        records.append({"reduction": f"tSNE(perp={perp})", "clustering": f"KMeans(k={k})", **score_clustering(emb, cl)})
    for mcs in [10, 25, 50]:
        cl = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=10).fit_predict(emb)
        records.append({"reduction": f"tSNE(perp={perp})", "clustering": f"HDBSCAN(mcs={mcs})", **score_clustering(emb, cl)})

sweep_df = pd.DataFrame(records)
sweep_df.to_csv(DATA_ROOT / "sweep_results.csv", index=False)
print(f"Saved {len(sweep_df)} configurations")
print(sweep_df.sort_values('ari', ascending=False).head(10)[['reduction','clustering','ari','nmi']])
print(f"\nCompare to best EfficientNet-B1 ARI from Part 3: {best_effnet['ari']:.3f}")

/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr

Saved 200 configurations
              reduction   clustering       ari       nmi
81   UMAP(nn=15,md=0.1)  KMeans(k=3)  0.315652  0.381676
73   UMAP(nn=15,md=0.0)  KMeans(k=3)  0.315652  0.381676
89   UMAP(nn=15,md=0.3)  KMeans(k=3)  0.298534  0.329721
97   UMAP(nn=15,md=0.5)  KMeans(k=3)  0.297304  0.316567
72   UMAP(nn=15,md=0.0)  KMeans(k=2)  0.244924  0.261628
88   UMAP(nn=15,md=0.3)  KMeans(k=2)  0.219269  0.253030
82   UMAP(nn=15,md=0.1)  KMeans(k=4)  0.219123  0.362854
74   UMAP(nn=15,md=0.0)  KMeans(k=4)  0.217616  0.358423
96   UMAP(nn=15,md=0.5)  KMeans(k=2)  0.213673  0.240599
186       tSNE(perp=30)  KMeans(k=4)  0.210786  0.326006

Compare to best EfficientNet-B1 ARI from Part 3: 0.157


## Part 5 — Class-balance sensitivity + per-treatment precision/recall

In [ ]:
best_embedding_full = umap.UMAP(n_neighbors=15, min_dist=0.5, random_state=42).fit_transform(features)
BEST_K = 3

rng = np.random.default_rng(42)
balanced_idx = []
for cond in CONDS:
    idx = np.where(treatment_labels == cond)[0]
    n = min(len(idx), min(pd.Series(treatment_labels).value_counts()))
    balanced_idx.extend(rng.choice(idx, size=n, replace=False))
balanced_idx = np.array(balanced_idx)

bal_features = features[balanced_idx]
bal_labels = treatment_labels[balanced_idx]
bal_embedding = umap.UMAP(n_neighbors=15, min_dist=0.5, random_state=42).fit_transform(bal_features)
bal_clusters = KMeans(n_clusters=BEST_K, random_state=42, n_init=10).fit_predict(bal_embedding)

balanced_crosstab = pd.crosstab(bal_labels, bal_clusters, margins=True)
print(balanced_crosstab)
print(f"\nBalanced ARI: {adjusted_rand_score(bal_labels, bal_clusters):.3f}")

ct = pd.crosstab(bal_labels, bal_clusters)
majority_per_cluster = ct.idxmax(axis=0)
print("\nPer-treatment recall/precision:")
for treatment in ct.index:
    claimed = majority_per_cluster[majority_per_cluster == treatment].index.tolist()
    if not claimed:
        print(f"  {treatment}: NEVER the majority in any cluster")
        continue
    tp = ct.loc[treatment, claimed].sum()
    recall = tp / ct.loc[treatment].sum()
    precision = tp / ct[claimed].values.sum()
    print(f"  {treatment}: claims {claimed} | recall={recall:.2f} precision={precision:.2f}")

/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


col_0   0   1   2  All
row_0                 
DMSO    4   1  22   27
TC04    0  20   7   27
TC11    8  10   9   27
TC39   20   0   7   27
All    32  31  45  108

Balanced ARI: 0.244

Per-treatment recall/precision:
  DMSO: claims [2] | recall=0.81 precision=0.49
  TC04: claims [1] | recall=0.74 precision=0.65
  TC11: NEVER the majority in any cluster
  TC39: claims [0] | recall=0.74 precision=0.62


## Part 6 — Semi-supervised embedding + held-out validation

In [ ]:
guidance_records = []
guidance_embeddings = {}

for weight in [0.0, 0.2, 0.4, 0.6, 0.8]:
    reducer = umap.UMAP(n_neighbors=20, min_dist=0.05, target_weight=weight, random_state=7)
    emb = reducer.fit_transform(features) if weight == 0.0 else reducer.fit_transform(features, y=treatment_ids)
    guidance_embeddings[weight] = emb
    hard = hdbscan.HDBSCAN(min_cluster_size=30, min_samples=10).fit_predict(emb)
    guidance_records.append({
        "target_weight": weight,
        "n_clusters_found": len(set(hard)) - (1 if -1 in hard else 0),
        "ari": adjusted_rand_score(treatment_ids, hard),
        "nmi": normalized_mutual_info_score(treatment_ids, hard),
    })

guidance_df = pd.DataFrame(guidance_records)
guidance_df.to_csv(DATA_ROOT / "semi_supervised_umap_sweep.csv", index=False)
print(guidance_df.to_string(index=False))

/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


 target_weight  n_clusters_found      ari      nmi
           0.0                 3 0.110700 0.268749
           0.2                 5 0.668433 0.816531
           0.4                 5 0.668433 0.816531
           0.6                 5 0.668433 0.816531
           0.8                 5 0.668433 0.816531


In [ ]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    np.arange(len(treatment_ids)), test_size=0.2, stratify=treatment_ids, random_state=42
)
y_partial = treatment_ids.copy().astype(float)
y_partial[test_idx] = -1

reducer = umap.UMAP(n_neighbors=20, min_dist=0.05, target_weight=0.4, random_state=7)
emb_holdout = reducer.fit_transform(features, y=y_partial)
predicted_labels = hdbscan.HDBSCAN(min_cluster_size=30, min_samples=10).fit_predict(emb_holdout)

holdout_ari = adjusted_rand_score(treatment_ids[test_idx], predicted_labels[test_idx])
full_guidance_ari = guidance_df.loc[guidance_df.target_weight == 0.4, "ari"].values[0]
print(f"Held-out ARI: {holdout_ari:.3f}")
print(f"Full-guidance (circular) ARI at same weight: {full_guidance_ari:.3f}")

/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Held-out ARI: 0.231
Full-guidance (circular) ARI at same weight: 0.668


## Part 7 — Hierarchical / soft clustering

In [ ]:
base_embedding = guidance_embeddings[0.0]
clusterer = hdbscan.HDBSCAN(min_cluster_size=30, min_samples=10, cluster_selection_method="eom", prediction_data=True)
hard_assignment = clusterer.fit_predict(base_embedding)
soft = hdbscan.all_points_membership_vectors(clusterer)
confidence = soft.max(axis=1)
transitional = confidence < 0.6

by_treatment = (pd.DataFrame({"treatment": treatment_labels, "transitional": transitional})
                .groupby("treatment")["transitional"].mean().sort_values(ascending=False))
print("Proportion transitional by treatment:")
print(by_treatment)
by_treatment.to_csv(DATA_ROOT / "transitional_fraction_by_treatment.csv")

plt.figure(figsize=(9, 6))
clusterer.condensed_tree_.plot(select_clusters=True, colorbar=True)
plt.title("Condensed tree: phenotype hierarchy")
plt.tight_layout()
plt.savefig(FIG_DIR / "hdbscan_condensed_tree.png", dpi=150)
plt.close()
print("Saved condensed tree")

pd.DataFrame({"treatment": treatment_labels, "hard_cluster": hard_assignment,
              "confidence": confidence, "transitional": transitional}
             ).to_csv(DATA_ROOT / "soft_cluster_assignments.csv", index=False)

Proportion transitional by treatment:
treatment
TC39    0.512821
TC11    0.274194
DMSO    0.257353
TC04    0.185185
Name: transitional, dtype: float64
Saved condensed tree


## Part 8 — Supervisor-requested diagnostics

In [ ]:
worst = sweep_df[(sweep_df.reduction.str.contains("tSNE")) & (sweep_df.clustering.str.contains("HDBSCAN"))]
worst = worst.sort_values("ari").iloc[0]
print("Worst t-SNE+HDBSCAN configuration:")
print(worst[["reduction", "clustering", "n_clusters_found", "pct_noise", "ari"]])

Worst t-SNE+HDBSCAN configuration:
reduction              tSNE(perp=5)
clustering          HDBSCAN(mcs=50)
n_clusters_found                  0
pct_noise                     100.0
ari                             0.0
Name: 175, dtype: object


In [ ]:
from sklearn.neighbors import NearestNeighbors

emb_02 = guidance_embeddings[0.2]
nn_model = NearestNeighbors(n_neighbors=15).fit(emb_02)
_, neighbour_idx = nn_model.kneighbors(emb_02)
same_label_frac = np.mean([
    np.mean(treatment_ids[neighbour_idx[i][1:]] == treatment_ids[i])
    for i in range(len(treatment_ids))
])
print(f"Average fraction of each cell's 15 nearest neighbours sharing its true label: {same_label_frac:.2f}")

Average fraction of each cell's 15 nearest neighbours sharing its true label: 0.98


## Part 9 — All figures

In [ ]:
plt.rcParams.update({"font.size": 11, "figure.dpi": 150})
rng = np.random.default_rng(5)

def load_cell_rgb_uint8(path, size=None):
    raw = skio.imread(str(path))
    if raw.ndim == 2:
        raw = raw[None, ...]
    ch0 = normalise_channel(raw[0])
    ch1 = normalise_channel(raw[1]) if raw.shape[0] > 1 else np.zeros_like(ch0)
    rgb = np.stack([np.zeros_like(ch0), ch0, ch1], axis=-1)
    if size is not None:
        rgb = np.array(Image.fromarray(rgb).resize(size))
    return rgb

# 9a: charts
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(guidance_df["target_weight"], guidance_df["ari"], "o-", color="#2a78d6", label="Full guidance (circular)")
ax.axhline(holdout_ari, color="#eb6834", linestyle="--", label=f"Held-out ARI = {holdout_ari:.3f}")
ax.axhline(guidance_df.loc[guidance_df.target_weight == 0.0, "ari"].values[0], color="gray", linestyle=":", label="Unsupervised ARI")
ax.set_xlabel("target_weight"); ax.set_ylabel("ARI"); ax.legend(fontsize=9); ax.set_ylim(-0.1, 1.0)
ax.set_title("Semi-supervised UMAP: apparent gain does not generalise")
plt.tight_layout(); plt.savefig(FIG_DIR / "3_semi_supervised_guidance_curve.png", bbox_inches="tight"); plt.close()

order = by_treatment.sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(order.index, order.values * 100, color=["#eb6834" if v == order.max() else "#2a78d6" for v in order.values])
for bar, val in zip(bars, order.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val * 100 + 0.5, f"{val*100:.1f}%", ha="center")
ax.set_ylabel("Transitional cells, %"); ax.set_title("Proportion of morphologically ambiguous cells by treatment")
plt.tight_layout(); plt.savefig(FIG_DIR / "4_transitional_fraction_by_treatment.png", bbox_inches="tight"); plt.close()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(ct.values, cmap="YlGnBu")
ax.set_xticks(range(len(ct.columns))); ax.set_xticklabels([f"Cluster {c}" for c in ct.columns])
ax.set_yticks(range(len(ct.index))); ax.set_yticklabels(ct.index)
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        ax.text(j, i, str(ct.values[i, j]), ha="center", va="center", color="white" if ct.values[i, j] > ct.values.max() / 2 else "black")
ax.set_title("Treatment vs. discovered cluster (balanced)"); plt.colorbar(im, label="Cell count")
plt.tight_layout(); plt.savefig(FIG_DIR / "5_balanced_crosstab_heatmap.png", bbox_inches="tight"); plt.close()
print("Saved 9a")

Saved 9a


In [ ]:
# 9b: confidence spectrum
def confidence_spectrum(n_each=4):
    conds = sorted(set(treatment_labels))
    fig, axes = plt.subplots(len(conds), n_each * 2 + 1, figsize=(1.9 * (n_each * 2 + 1), 2.1 * len(conds)))
    for row, cond in enumerate(conds):
        idx = np.where(treatment_labels == cond)[0]
        order_idx = idx[np.argsort(-confidence[idx])]
        seq = list(order_idx[:n_each]) + [None] + list(order_idx[-n_each:])
        for col, ci in enumerate(seq):
            ax = axes[row, col]; ax.axis("off")
            if ci is None: continue
            ax.imshow(load_cell_rgb_uint8(cell_index[ci]["crop_path"]))
            ax.set_title(f"{confidence[ci]:.2f}", fontsize=8)
        axes[row, 0].set_ylabel(cond, rotation=0, labelpad=35, fontsize=12, fontweight="bold")
        axes[row, 0].axis("on"); axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
        for s in axes[row, 0].spines.values(): s.set_visible(False)
    plt.tight_layout(); plt.savefig(FIG_DIR / "A_confidence_spectrum.png", bbox_inches="tight", dpi=150); plt.close()
    print("Saved 9b")

confidence_spectrum()

Saved 9b


In [ ]:
# 9c: 3D tree
from collections import defaultdict

num_points = len(features)
ct_tree = clusterer.condensed_tree_.to_pandas()
cluster_edges = ct_tree[ct_tree["child"] >= num_points][["parent", "child", "lambda_val"]]
children_of, lambda_of_edge = defaultdict(list), {}
for _, row in cluster_edges.iterrows():
    p, c, lam = int(row["parent"]), int(row["child"]), float(row["lambda_val"])
    children_of[p].append(c); lambda_of_edge[(p, c)] = lam
all_parents, all_children = set(cluster_edges["parent"]), set(cluster_edges["child"])
root = min(all_parents - all_children) if (all_parents - all_children) else int(cluster_edges["parent"].min())

positions, next_x = {}, [0.0]
def layout(node, z):
    kids = children_of.get(node, [])
    if not kids:
        x = next_x[0]; next_x[0] += 1.0; positions[node] = (x, z); return x
    xs = [layout(k, lambda_of_edge[(node, k)]) for k in kids]
    x = sum(xs) / len(xs); positions[node] = (x, z); return x
layout(root, 0.0)
max_z = max(z for _, z in positions.values()) or 1.0

fig = plt.figure(figsize=(10, 8)); ax = fig.add_subplot(111, projection="3d")
def draw(node):
    x, z = positions[node]
    for k in children_of.get(node, []):
        xk, zk = positions[k]
        y0, y1 = np.sin(x * 0.7) * 0.5, np.sin(xk * 0.7) * 0.5
        color = plt.cm.copper(1 - z / max_z)
        ax.plot([x, xk], [y0, y1], [z, zk], color=color, linewidth=max(0.6, 3 - 2 * (z / max_z)))
        draw(k)
draw(root)
leaf_nodes = [n for n in positions if not children_of.get(n)]
for n in leaf_nodes:
    x, z = positions[n]; y = np.sin(x * 0.7) * 0.5
    ax.scatter([x], [y], [z], color="seagreen", s=70, edgecolor="darkgreen")
ax.set_title("Phenotype hierarchy as a branching tree")
ax.set_xlabel("branch position"); ax.set_zlabel("split depth (lambda)")
ax.view_init(elev=20, azim=35)
plt.tight_layout(); plt.savefig(FIG_DIR / "D_3d_phenotype_tree.png", dpi=150, bbox_inches="tight"); plt.close()
print(f"Saved 9c ({len(leaf_nodes)} branch tips)")

Saved 9c (4 branch tips)


In [ ]:
# 9d: sanity check + cluster montage
sample = rng.choice(len(cell_index), size=min(8, len(cell_index)), replace=False)
fig, axes = plt.subplots(1, len(sample), figsize=(2.2 * len(sample), 2.4))
for ax, i in zip(axes, sample):
    rec = cell_index[i]
    ax.imshow(load_cell_rgb_uint8(rec["crop_path"]))
    ax.axis("off"); ax.set_title(rec["treatment"], fontsize=9)
fig.suptitle("Sample isolated cells (green: tubulin, blue: DAPI)", y=1.05)
plt.tight_layout(); plt.savefig(FIG_DIR / "G_isolated_cell_sanity_check.png", bbox_inches="tight", dpi=150); plt.close()
print("Saved 9d-i")

unique_clusters = sorted(set(hard_assignment) - {-1})
n_per = 5
fig, axes = plt.subplots(len(unique_clusters), n_per, figsize=(2.4 * n_per, 2.4 * len(unique_clusters)))
for row, cl in enumerate(unique_clusters):
    idx = np.where(hard_assignment == cl)[0]
    chosen = rng.choice(idx, size=min(n_per, len(idx)), replace=False)
    for col in range(n_per):
        ax = axes[row, col] if len(unique_clusters) > 1 else axes[col]
        if col >= len(chosen):
            ax.axis("off"); continue
        ax.imshow(load_cell_rgb_uint8(cell_index[chosen[col]]["crop_path"]))
        ax.axis("off")
    label_ax = axes[row, 0] if len(unique_clusters) > 1 else axes[0]
    label_ax.axis("on"); label_ax.set_xticks([]); label_ax.set_yticks([])
    for s in label_ax.spines.values(): s.set_visible(False)
    label_ax.set_ylabel(f"Cluster {cl}", rotation=0, labelpad=45, fontsize=12, fontweight="bold")
fig.suptitle("Representative isolated cells by cluster", y=1.01)
plt.tight_layout(); plt.savefig(FIG_DIR / "H_cluster_montage.png", bbox_inches="tight", dpi=150); plt.close()
print("Saved 9d-ii")

Saved 9d-i
Saved 9d-ii


In [ ]:
# 9e: image-scatter embedding
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.patches import Rectangle

treatment_colors = {t: c for t, c in zip(sorted(set(treatment_labels)), plt.cm.tab10.colors)}
emb = guidance_embeddings[0.0]
idx_sample = rng.choice(len(emb), size=min(90, len(emb)), replace=False)

fig, ax = plt.subplots(figsize=(11, 9))
ax.scatter(emb[:, 0], emb[:, 1], s=4, color="lightgray", alpha=0.4, zorder=1)
for i in idx_sample:
    rgb_thumb = load_cell_rgb_uint8(cell_index[i]["crop_path"], size=(60, 60))
    color = tuple(int(255 * c) for c in treatment_colors[treatment_labels[i]][:3])
    img = ImageOps.expand(Image.fromarray(rgb_thumb), border=4, fill=color)
    ab = AnnotationBbox(OffsetImage(np.array(img), zoom=0.35), (emb[i, 0], emb[i, 1]), frameon=False, zorder=2)
    ax.add_artist(ab)
handles = [Rectangle((0, 0), 1, 1, facecolor=treatment_colors[t], edgecolor="black") for t in sorted(set(treatment_labels))]
ax.legend(handles, sorted(set(treatment_labels)), loc="upper right", fontsize=9, title="True treatment")
ax.set_title("Phenotype embedding with representative cell images\n(border colour = TRUE TREATMENT; grey = full population)")
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.savefig(FIG_DIR / "F_image_scatter_embedding_truelabels.png", bbox_inches="tight", dpi=180); plt.close()
print("Saved 9e")
print("\n\u2713 All Part 9 figures complete.")

Saved 9e

✓ All Part 9 figures complete.


## Part 10 — Export everything

In [ ]:
shutil.make_archive("/content/all_results", "zip", DATA_ROOT)
shutil.make_archive("/content/all_figures", "zip", FIG_DIR)
print("Saved /content/all_results.zip and /content/all_figures.zip")
print("Download both via the Colab file browser.")

Saved /content/all_results.zip and /content/all_figures.zip
Download both via the Colab file browser.


In [ ]:
# ============================================================
# Stability check: is the "which treatment is most transitional"
# finding consistent across runs, or is it noise at n=264?
# Re-runs just the embedding + HDBSCAN + soft-clustering step
# (not the whole pipeline) with 5 different random seeds.
# ============================================================
import hdbscan
import umap
import pandas as pd

stability_results = []

for seed in [0, 1, 2, 3, 4]:
    emb = umap.UMAP(n_neighbors=20, min_dist=0.05, random_state=seed).fit_transform(features)
    clusterer_run = hdbscan.HDBSCAN(min_cluster_size=30, min_samples=10,
                                      cluster_selection_method="eom", prediction_data=True)
    hard = clusterer_run.fit_predict(emb)
    soft = hdbscan.all_points_membership_vectors(clusterer_run)
    conf = soft.max(axis=1)
    trans = conf < 0.6

    by_treat = (pd.DataFrame({"treatment": treatment_labels, "transitional": trans})
                .groupby("treatment")["transitional"].mean())

    row = {"seed": seed}
    row.update(by_treat.to_dict())
    row["highest"] = by_treat.idxmax()
    row["lowest"] = by_treat.idxmin()
    stability_results.append(row)

stability_df = pd.DataFrame(stability_results)
print("Transitional-cell proportion by treatment, across 5 random seeds:")
print(stability_df.to_string(index=False))

print(f"\n'Highest transitional' treatment across seeds: {stability_df['highest'].value_counts().to_dict()}")
print(f"'Lowest transitional' treatment across seeds: {stability_df['lowest'].value_counts().to_dict()}")

if stability_df["highest"].nunique() == 1:
    print("\n-> STABLE: same treatment is highest across all seeds.")
else:
    print("\n-> UNSTABLE: which treatment ranks highest changes with the random seed.")
    print("   This should be reported as a limitation -- the hierarchical/soft-clustering")
    print("   finding is sensitive to embedding stochasticity at this sample size,")
    print("   rather than presented as a single settled result.")

stability_df.to_csv(DATA_ROOT / "transitional_stability_check.csv", index=False)

/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Transitional-cell proportion by treatment, across 5 random seeds:
 seed     DMSO     TC04     TC11     TC39 highest lowest
    0 0.066176 0.000000 0.112903 0.051282    TC11   TC04
    1 0.051471 0.037037 0.048387 0.000000    DMSO   TC39
    2 0.058824 0.037037 0.032258 0.000000    DMSO   TC39
    3 0.058824 0.000000 0.032258 0.000000    DMSO   TC04
    4 0.073529 0.851852 0.290323 0.051282    TC04   TC39

'Highest transitional' treatment across seeds: {'DMSO': 3, 'TC11': 1, 'TC04': 1}
'Lowest transitional' treatment across seeds: {'TC39': 3, 'TC04': 2}

-> UNSTABLE: which treatment ranks highest changes with the random seed.
   This should be reported as a limitation -- the hierarchical/soft-clustering
   finding is sensitive to embedding stochasticity at this sample size,
   rather than presented as a single settled result.


In [ ]:
# ============================================================
# DINOv2 comparison -- actually implementing the "future work"
# direction motivated in Section 2.4, rather than leaving it
# proposed-but-untested. Same structure as the EfficientNet-B1
# comparison (Part 3), so this should run with no surprises.
# ============================================================
!pip install -q transformers

import torch
from transformers import AutoImageProcessor, AutoModel
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import umap
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# facebook/dinov2-small: smallest official DINOv2 checkpoint, fastest to run
processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
dino_model = AutoModel.from_pretrained("facebook/dinov2-small").to(device).eval()

def load_rgb_pil(path):
    raw = skio.imread(str(path))
    if raw.ndim == 2:
        raw = raw[None, ...]
    ch0 = normalise_channel(raw[0])
    ch1 = normalise_channel(raw[1]) if raw.shape[0] > 1 else np.zeros_like(ch0)
    rgb = np.stack([np.zeros_like(ch0), ch0, ch1], axis=-1)
    return Image.fromarray(rgb)

crop_paths = [c["crop_path"] for c in cell_index]
dino_features = []

with torch.no_grad():
    for start in tqdm(range(0, len(crop_paths), 16), desc="DINOv2 features"):
        batch_paths = crop_paths[start:start + 16]
        images = [load_rgb_pil(p) for p in batch_paths]
        inputs = processor(images=images, return_tensors="pt").to(device)
        outputs = dino_model(**inputs)
        # CLS token embedding: the standard DINOv2 image-level representation
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        dino_features.append(cls_embeddings)

dino_features = np.concatenate(dino_features, axis=0)
print(f"DINOv2 feature matrix: {dino_features.shape}")

# Same sweep structure as EfficientNet-B1 comparison, for a fair three-way comparison
dino_records = []
for nn in [5, 15, 30, 50]:
    for md_ in [0.0, 0.1, 0.3, 0.5]:
        emb = umap.UMAP(n_neighbors=nn, min_dist=md_, random_state=42).fit_transform(dino_features)
        for k in [2, 3, 4, 6, 8]:
            cl = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(emb)
            dino_records.append({
                "nn": nn, "md": md_, "k": k,
                "ari": adjusted_rand_score(treatment_ids, cl),
                "nmi": normalized_mutual_info_score(treatment_ids, cl),
            })

dino_df = pd.DataFrame(dino_records)
best_dino = dino_df.sort_values("ari", ascending=False).iloc[0]
print("\nTop 5 DINOv2 configurations:")
print(dino_df.sort_values("ari", ascending=False).head(5))
print(f"\nBest DINOv2 ARI: {best_dino['ari']:.3f} (nn={best_dino['nn']}, md={best_dino['md']}, k={best_dino['k']})")

print("\n=== Three-way feature extractor comparison ===")
print(f"ResNet50 (supervised CNN):       0.316")
print(f"EfficientNet-B1 (supervised CNN): 0.157")
print(f"DINOv2-small (self-supervised ViT): {best_dino['ari']:.3f}")

dino_df.to_csv(DATA_ROOT / "dino_sweep_results.csv", index=False)
np.save(DATA_ROOT / "dino_features.npy", dino_features)

Using device: cpu


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

DINOv2 features:   0%|          | 0/17 [00:00<?, ?it/s]

DINOv2 feature matrix: (264, 384)


/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr


Top 5 DINOv2 configurations:
    nn   md  k       ari       nmi
70  50  0.3  2  0.253805  0.298985
25  15  0.1  2  0.249917  0.297413
5    5  0.1  2  0.243291  0.329920
50  30  0.3  2  0.243291  0.329920
20  15  0.0  2  0.243291  0.329920

Best DINOv2 ARI: 0.254 (nn=50.0, md=0.3, k=2.0)

=== Three-way feature extractor comparison ===
ResNet50 (supervised CNN):       0.316
EfficientNet-B1 (supervised CNN): 0.157
DINOv2-small (self-supervised ViT): 0.254


In [ ]:
# ============================================================
# Real 3D visualization of the actual feature space -- computed
# from your genuine ResNet50 features, not illustrative/fabricated
# coordinates. Multi-angle panel (a single static 3D view can be
# misleading, so this shows 3 rotations of the same real embedding).
#
# Run this in the same Colab session as your main pipeline (needs
# features, treatment_labels, hard_assignment already in memory).
# ============================================================

import umap
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Real 3-component UMAP embedding of your actual ResNet50 features
# (same features used throughout the dissertation, just projected to
# 3D instead of 2D for this visualization)
embedding_3d = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=3,
                          random_state=42).fit_transform(features)

treatment_colors = {
    "DMSO": "#185FA5",
    "TC04": "#534AB7",
    "TC11": "#BA7517",   # distinct colour, since TC11 is the finding of interest
    "TC39": "#993C1D",
}
treatment_markers = {
    "DMSO": "o", "TC04": "o", "TC39": "o",
    "TC11": "D",  # diamond marker -- visually sets TC11 apart, matching
                   # the "no distinct territory" framing, but with REAL coordinates
}

fig = plt.figure(figsize=(16, 5.5))
angles = [(20, 30), (20, 120), (20, 210)]  # three rotations of the SAME real embedding

for i, (elev, azim) in enumerate(angles):
    ax = fig.add_subplot(1, 3, i + 1, projection="3d")
    for treatment in ["DMSO", "TC04", "TC39", "TC11"]:  # TC11 drawn last so it's visible on top
        mask = treatment_labels == treatment
        ax.scatter(
            embedding_3d[mask, 0], embedding_3d[mask, 1], embedding_3d[mask, 2],
            c=treatment_colors[treatment],
            marker=treatment_markers[treatment],
            s=55 if treatment == "TC11" else 40,
            alpha=0.85,
            edgecolors="white", linewidths=0.4,
            label=treatment if i == 0 else None,
        )
    ax.view_init(elev=elev, azim=azim)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.set_xlabel("UMAP-1", fontsize=9); ax.set_ylabel("UMAP-2", fontsize=9); ax.set_zlabel("UMAP-3", fontsize=9)
    ax.set_title(f"View {i+1} (elev={elev}°, azim={azim}°)", fontsize=10)

fig.legend(loc="lower center", ncol=4, fontsize=11, frameon=False, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(FIG_DIR / "I_3d_embedding_multiangle.png", dpi=180, bbox_inches="tight")
plt.close()
print("Saved I_3d_embedding_multiangle.png")
print("\nThis is a REAL 3D UMAP embedding of your actual ResNet50 features --")
print("not a schematic. TC11 (diamonds) can be visually inspected across all")
print("three rotations to check whether it clusters with any other treatment")
print("or genuinely disperses, exactly as the balanced crosstab quantifies.")

/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved I_3d_embedding_multiangle.png

This is a REAL 3D UMAP embedding of your actual ResNet50 features --
not a schematic. TC11 (diamonds) can be visually inspected across all
three rotations to check whether it clusters with any other treatment
or genuinely disperses, exactly as the balanced crosstab quantifies.
